# 04b — Model Comparison: Ridge, RF, CatBoost, Ensemble

**Goal**: Compare alternative models for Agent 2 (Conversion Predictor) when CatBoost shows weak signal (ROC-AUC ~0.50).

**Research basis**:
- Ridge regression & Random Forest handle weak signals better than LASSO/GBDT (Shen & Xiu, Chicago Booth 2024)
- Extended feature engineering (ratio-like interactions) can improve AUC

**Models**: Ridge (L2), Random Forest, CatBoost (baseline), Ensemble

## 1. Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import os

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
sns.set_theme(style="whitegrid")

MODELS_DIR = Path("../models")
train_df = pd.read_parquet(MODELS_DIR / "train.parquet")
test_df = pd.read_parquet(MODELS_DIR / "test.parquet")
feature_config = joblib.load(MODELS_DIR / "feature_config.joblib")

BASE_FEATURES = feature_config["AGENT2_FEATURES"]
TARGET = "Policy_Bind_enc"

print(f"Base features: {len(BASE_FEATURES)}")
print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(f"Bind rate: {train_df[TARGET].mean():.2%}")

Base features: 18
Train: 117007, Test: 29252
Bind rate: 22.22%


## 2. Extended Feature Engineering

Add ratio-like and interaction features (research: explicit ratios improve AUC when trees struggle to synthesize them):
- `affordability_ratio`: salary vs coverage
- `hh_need`: household driver demand
- `risk_coverage_fit`: risk × coverage
- `premium_to_salary`: Quoted_Premium / (salary band proxy)
- `log_premium_ratio`: log(1 + premium) for scale
- `risk_prev_interaction`: risk_tier × (accidents + citations)
- `urgency_coverage`: urgency_days × coverage level

In [2]:
def add_extended_features(df: pd.DataFrame) -> pd.DataFrame:
    """Extended feature engineering for weak-signal improvement."""
    out = df.copy()
    eps = 1e-6

    # Original domain features
    if "Sal_Range_enc" in out.columns and "Coverage_enc" in out.columns:
        out["affordability_ratio"] = (out["Sal_Range_enc"] / 4.0) - (out["Coverage_enc"] / 2.0)
    if "HH_Drivers" in out.columns:
        out["hh_need"] = np.clip(out["HH_Drivers"] - 1, 0, 3) / 3.0
    if "risk_tier_enc" in out.columns and "Coverage_enc" in out.columns:
        out["risk_coverage_fit"] = out["risk_tier_enc"] + out["Coverage_enc"]

    # Ratio-like: premium vs salary band (higher band = more income)
    if "Quoted_Premium" in out.columns and "Sal_Range_enc" in out.columns:
        sal_proxy = (out["Sal_Range_enc"] + 1) * 10000  # proxy for income
        out["premium_to_salary"] = np.log1p(out["Quoted_Premium"] / (sal_proxy + eps))

    # Log premium (scale normalization)
    if "Quoted_Premium" in out.columns:
        out["log_premium"] = np.log1p(out["Quoted_Premium"])

    # Risk × prior incidents interaction
    if "risk_tier_enc" in out.columns and "Prev_Accidents" in out.columns and "Prev_Citations" in out.columns:
        out["risk_prev_interaction"] = out["risk_tier_enc"] * (out["Prev_Accidents"] + out["Prev_Citations"] + 1)

    # Urgency × coverage (higher coverage + more urgency = different conversion pattern)
    if "urgency_days" in out.columns and "Coverage_enc" in out.columns:
        out["urgency_coverage"] = (out["urgency_days"] / 30.0) * (out["Coverage_enc"] + 1)

    # Re-quote × risk (suspicious pattern)
    if "Re_Quote_enc" in out.columns and "risk_tier_enc" in out.columns:
        out["requote_risk"] = out["Re_Quote_enc"] * out["risk_tier_enc"]

    return out


train_df = add_extended_features(train_df)
test_df = add_extended_features(test_df)

EXTRA_FEATURES = [
    c for c in [
        "affordability_ratio", "hh_need", "risk_coverage_fit",
        "premium_to_salary", "log_premium", "risk_prev_interaction",
        "urgency_coverage", "requote_risk"
    ] if c in train_df.columns
]
FEATURES = BASE_FEATURES + EXTRA_FEATURES

X_train = train_df[FEATURES].fillna(0).copy()
y_train = train_df[TARGET].copy()
X_test = test_df[FEATURES].fillna(0).copy()
y_test = test_df[TARGET].copy()

print(f"Total features: {len(FEATURES)}")
print(f"Extra: {EXTRA_FEATURES}")

Total features: 26
Extra: ['affordability_ratio', 'hh_need', 'risk_coverage_fit', 'premium_to_salary', 'log_premium', 'risk_prev_interaction', 'urgency_coverage', 'requote_risk']


## 3. Scale Features & SMOTE

In [3]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE: {X_train_sm.shape[0]} rows")
print(f"Distribution: {pd.Series(y_train_sm).value_counts().to_dict()}")

After SMOTE: 182010 rows
Distribution: {0: 91005, 1: 91005}


## 4. Train Models

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

def eval_model(name, y_prob, y_true):
    return {
        "model": name,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }

results = []
models = {}
probs = {}

# --- Ridge (L2 logistic regression) ---
ridge = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=42)
ridge.fit(X_train_sm, y_train_sm)
probs["ridge"] = ridge.predict_proba(X_test_scaled)[:, 1]
results.append(eval_model("Ridge", probs["ridge"], y_test))
models["ridge"] = ridge
print("Ridge: done")

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
probs["rf"] = rf.predict_proba(X_test_scaled)[:, 1]
results.append(eval_model("RandomForest", probs["rf"], y_test))
models["rf"] = rf
print("Random Forest: done")

# --- CatBoost (on scaled data; CatBoost handles raw too but we use same pipeline) ---
from catboost import CatBoostClassifier
cb = CatBoostClassifier(iterations=500, depth=6, learning_rate=0.03, loss_function="Logloss",
                       auto_class_weights="Balanced", random_seed=42, verbose=0)
cb.fit(X_train_sm, y_train_sm)
probs["catboost"] = cb.predict_proba(X_test_scaled)[:, 1]
results.append(eval_model("CatBoost", probs["catboost"], y_test))
models["catboost"] = cb
print("CatBoost: done")

Ridge: done
Random Forest: done
CatBoost: done


## 5. Ensemble

In [5]:
# Weighted average of probabilities (equal weight)
results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
print("=== Model Comparison ===")
print(results_df.to_string(index=False))

top_models = results_df.head(3)["model"].str.lower().tolist()
top_map = {"ridge": "ridge", "randomforest": "rf", "catboost": "catboost"}
ensemble_keys = [top_map.get(m, m) for m in top_models if top_map.get(m, m) in probs]

if len(ensemble_keys) >= 2:
    ensemble_prob = np.mean([probs[k] for k in ensemble_keys], axis=0)
    probs["ensemble"] = ensemble_prob
    results.append(eval_model("Ensemble", ensemble_prob, y_test))
    print(f"\nEnsemble: avg of {ensemble_keys}")

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
print("\n=== Final Ranking ===")
print(results_df.to_string(index=False))

=== Model Comparison ===
       model  roc_auc   pr_auc
    CatBoost 0.502091 0.222327
       Ridge 0.501910 0.221480
RandomForest 0.496308 0.220287

Ensemble: avg of ['catboost', 'ridge', 'rf']

=== Final Ranking ===
       model  roc_auc   pr_auc
    CatBoost 0.502091 0.222327
       Ridge 0.501910 0.221480
    Ensemble 0.498848 0.219546
RandomForest 0.496308 0.220287


## 6. Select Best & Save

In [6]:
pass  # Ensemble computed above

### Calibration & Persist

In [7]:
best_row = results_df.iloc[0]
best_name = best_row["model"].lower()
best_map = {"ridge": "ridge", "randomforest": "rf", "catboost": "catboost", "ensemble": "ensemble"}
best_key = best_map.get(best_name, "ridge")

y_prob_best = probs[best_key]
print(f"Best model: {best_row['model']} (ROC-AUC={best_row['roc_auc']:.4f}, PR-AUC={best_row['pr_auc']:.4f})")

# --- Premium Advisor Threshold (used by pipeline graph for Agent 3 routing) ---
# With weak signal (ROC-AUC ~0.50), use fixed 60; pipeline loads from bundle.
RECOMMENDED_PA_THRESHOLD = 60
print(f"Premium advisor threshold: {RECOMMENDED_PA_THRESHOLD} (pipeline loads from bundle)")

# Save model and pipeline artifacts
joblib.dump(scaler, MODELS_DIR / "conversion_scaler.joblib")
joblib.dump(add_extended_features, MODELS_DIR / "add_extended_features_fn.joblib")

if best_key == "ensemble":
    ens_cfg = {"ensemble_keys": ensemble_keys, "scaler": scaler, "add_extended_features": add_extended_features}
    joblib.dump(ens_cfg, MODELS_DIR / "conversion_ensemble_config.joblib")
    for k in ensemble_keys:
        joblib.dump(models[k], MODELS_DIR / f"conversion_{k}.joblib")
    print("Saved: ensemble config + sub-models")
else:
    joblib.dump(models[best_key], MODELS_DIR / "conversion_model.joblib")
    print(f"Saved: conversion_model.joblib ({best_key})")

# Save feature list and best model name for backend
bundle = {
    "feature_names": FEATURES,
    "best_model": best_key,
    "best_roc_auc": float(best_row["roc_auc"]),
    "best_pr_auc": float(best_row["pr_auc"]),
    "premium_advisor_threshold": RECOMMENDED_PA_THRESHOLD,
    "results": results_df.to_dict(orient="records"),
}
joblib.dump(bundle, MODELS_DIR / "conversion_model_comparison_bundle.joblib")
print("Saved: conversion_model_comparison_bundle.joblib")

Best model: CatBoost (ROC-AUC=0.5021, PR-AUC=0.2223)
Premium advisor threshold (from calibration): 60
Saved: conversion_model.joblib (catboost)
Saved: conversion_model_comparison_bundle.joblib


## 8. Classification Report (Best Model)

In [8]:
from sklearn.metrics import classification_report

y_pred = (y_prob_best >= 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=["No Bind", "Bind"]))

              precision    recall  f1-score   support

     No Bind       0.78      1.00      0.88     22752
        Bind       0.00      0.00      0.00      6500

    accuracy                           0.78     29252
   macro avg       0.39      0.50      0.44     29252
weighted avg       0.60      0.78      0.68     29252



## Next Steps

If the best model from this notebook outperforms the CatBoost from notebook 04, update `Backend/agents/agent2_conversion_predictor.py` to:
1. Load `conversion_scaler.joblib` and `add_extended_features_fn.joblib`
2. Use the saved model (Ridge/RF/CatBoost/Ensemble) for prediction
3. Scale features before prediction for Ridge/RF/CatBoost